# Comparing embeddings

**What you'll learn.** When several models embed the same entities, you need to
know whether they encode the same structure — and *which* metric answers your
question. embpy ships a family of comparison tools; this notebook walks the main
ones and ends with a table for choosing between them.

Comparisons split into three kinds:

| Kind | Question | Tools |
| --- | --- | --- |
| **Global geometry** | is the overall shape the same? | `tsi`, `qsi`, `linear_cka`, `similarity_correlation` |
| **Local neighbourhoods** | are the same things nearby? | `compute_knn_overlap`, `knn_jaccard`, `mutual_knn` |
| **Visual** | what does it look like? | `cross_model_similarity`, `embedding_norms`, `plot_embedding_space` |

A model pair can score high globally and low locally, which is itself a finding.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd

from embpy import BioEmbedder, tl, pl

embedder = BioEmbedder(device="auto", organism="human")

genes = ["TP53", "EGFR", "MYC", "BRCA1", "JUN", "STAT1", "IRF1", "CDK1",
         "GATA3", "FOXP3", "CD8A", "IL2", "CCND1", "RB1", "PTEN", "AKT1"]
adata = ad.AnnData(
    X=np.zeros((len(genes), 1), dtype=np.float32),
    obs=pd.DataFrame({"symbol": genes}, index=genes),
)

## Embed the same genes three ways

A prior-knowledge table, a co-expression vector, and a protein language model —
three genuinely different views of the same 16 genes.

In [ ]:
specs = [("genept", "gene", "X_prior"),
         ("gene2vec", "gene", "X_coexp"),
         ("esm2_8M", "protein", "X_esm2")]

for model, entity, key in specs:
    adata = embedder.embed(
        adata, entity_type=entity, id_type="symbol", obs_column="symbol",
        model=model, output="anndata", key=key, attach_to="obs",
    )

spaces = ["X_prior", "X_coexp", "X_esm2"]
embeddings = {k.replace("X_", ""): adata.obsm[k] for k in spaces}
print({k: v.shape for k, v in embeddings.items()})

## 1. Global geometry

`alignment_matrix` applies one metric to every pair. **TSI** is the one to reach
for first: it compares distance *orderings*, so its null is a fixed **0.5** and a
score reads the same on any dataset.

In [ ]:
tl.alignment_matrix(embeddings, metric="tsi").round(3)

Swap the metric to see where they disagree. **QSI** probes global geometry (no
shared anchor), **CKA** compares inner-product structure, **mutual_knn** is purely
local. Disagreement between them is diagnostic, not noise.

In [ ]:
summary = pd.DataFrame({
    m: tl.alignment_matrix(embeddings, metric=m).where(
        np.triu(np.ones((3, 3)), k=1).astype(bool)).stack()
    for m in ["tsi", "qsi", "cka", "mutual_knn"]
})
summary.round(3)

`similarity_correlation` asks a related question directly: do the two spaces rank
*pairwise distances* the same way?

In [ ]:
tl.similarity_correlation(embeddings["prior"], matrix_b=embeddings["esm2"],
                         label_a="prior_vs_esm2").round(3)

## 2. Local neighbourhoods

Global agreement can hide local disagreement. These ask whether each gene keeps
the *same neighbours*.

In [ ]:
k = 4
_, mean_overlap = tl.compute_knn_overlap(adata, "X_prior", "X_esm2", k=k)
_, jac = tl.knn_jaccard(embeddings["prior"], embeddings["esm2"], k=k)
mk = tl.mutual_knn(embeddings["prior"], embeddings["esm2"], k=k)

print(f"{'kNN overlap':<16}{mean_overlap:.3f}")
print(f"{'Jaccard':<16}{jac:.3f}")
print(f"{'mutual kNN':<16}{mk:.3f}")
print(f"\nchance level ~ k/(n-1) = {k/(len(genes)-1):.3f}")

`compare_embedding_matrices` runs a standard battery across every pair at once —
the quickest way to get a full picture.

In [ ]:
tl.compare_embedding_matrices(embeddings, k=k).round(3)

## 3. Visual comparison

In [ ]:
pl.cross_model_similarity(adata, obsm_keys=spaces)
pl.knn_overlap(adata, obsm_keys=spaces, k=k)

Embeddings live at wildly different magnitudes — worth seeing before you ever
concatenate or average them.

In [ ]:
pl.embedding_norms(adata, obsm_keys=spaces)
pl.embedding_distributions(adata, obsm_keys=spaces, n_dims=6)

In [ ]:
for key in spaces:
    pl.plot_embedding_space(adata, obsm_key=key, method="pca",
                            annotate=True, annotate_col="symbol", title=key)

## 4. The outlier problem

The decisive difference between the metric families. Two representations identical
up to rotation, with **2% of rows corrupted**:

In [ ]:
rng = np.random.default_rng(0)
n, d = 400, 32
X = rng.normal(size=(n, d))
Q, _ = np.linalg.qr(rng.normal(size=(d, d)))
Y = X @ Q                                  # true similarity is exactly 1.0
bad = rng.choice(n, size=int(0.02 * n), replace=False)
Y[bad] += rng.normal(scale=200.0, size=(len(bad), d))

print(f"TSI        {tl.tsi(X, Y):.3f}   <- barely moves")
print(f"linear CKA {tl.linear_cka(X, Y):.3f}   <- collapses")

CKA calls two rotation-identical spaces unrelated, because its Frobenius norms are
dominated by a handful of rows. Real single-cell data always has such rows — dying
cells, doublets, ambient RNA — so prefer the ordinal metrics there.

## 5. Which layer?

The final layer is specialised toward the pretraining objective and is routinely
*not* the best for transfer. `rank_layers` scores them from data.

In [ ]:
layers = {}
for L in range(7):                     # esm2_8M: embedding layer + 6 blocks
    out = embedder.embed(
        adata.copy(), entity_type="protein", id_type="symbol", obs_column="symbol",
        model="esm2_8M", layer=L, output="anndata", key="X", attach_to="obs",
    )
    layers[L] = out.obsm["X"]

tl.rank_layers(layers).round(3)

## Choosing a metric

| Metric | Scope | Null | Outlier-robust | Use when |
| --- | --- | --- | --- | --- |
| `tsi` | local (anchored) | **0.5** | yes | default; comparable across datasets |
| `qsi` | global | **0.5** | yes | overall geometry, no anchor |
| `linear_cka` | global | varies | **no** | comparing to published CKA numbers |
| `mutual_knn` | local | ~k/(n−1) | yes | neighbour sets only, ignores order |
| `knn_jaccard` | local | ~k/(2n−k) | yes | set overlap with a familiar statistic |
| `similarity_correlation` | global | 0 | no | correlating pairwise distances |

Rules of thumb: **TSI first** (fixed null, robust); add **QSI** if global structure
matters; use **CKA** only for comparability with the literature; reach for the
kNN family when neighbourhoods are what you act on.

All of these accept `metric="cosine"` or `"correlation"`, which is usually more
meaningful than Euclidean in high dimensions.

**Next:** [Which model captures my biology?](04_benchmark_models.ipynb) turns this
structural comparison into a task-specific score. The per-modality notebooks
([genes](genes.ipynb), [proteins](proteins.ipynb), [cells](cells.ipynb),
[small molecules](small_molecules.ipynb)) apply these metrics across every model in
their family.